In [ ]:
print("hi")

In [ ]:
import os, json, sys
print('python', sys.version)
try:
    import ollama
    import chromadb
    print('ollama ok', ollama)
    print('chromadb', chromadb.__version__)
except Exception as e:
    print('IMPORT_ERR', repr(e))
    sys.exit(0)

PERSISTENT_DIR = '../VectorDB/APIembeddings_db_storage'
collection_name='api_spec_cards'
client = chromadb.PersistentClient(path=PERSISTENT_DIR)
collection = client.get_or_create_collection(name=collection_name)
print('count', collection.count())
print('peek', collection.peek())

In [83]:
#persistent chromadb

import os
import json
import ollama
import chromadb
from typing import Dict, List

# Configuration constants
OPENAPI_FILE_PATH = "test.json"
EMBEDDING_MODEL = "nomic-embed-text"
COLLECTION_NAME = "api_spec_cards"

# --- PERSISTENCE UPDATE ---
# This directory will be automatically created on your machine to save database files.
PERSISTENT_DIR = "../VectorDB/APIembeddings_db_storage1"

# Initialize Persistent Storage Client instead of an in-memory client
chroma_client = chromadb.PersistentClient(path=PERSISTENT_DIR)

# Get or create collection. Using 'get_or_create' ensures we don't wipe existing disk files on rerun.
collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)


def get_embedding(text: str) -> List[float]:
    """Generate high-density text embeddings using your local Ollama client."""
    response = ollama.embeddings(model=EMBEDDING_MODEL, prompt=text)
    return response["embedding"]


def search_openapi_catalog(user_query: str):
    """Executes a similarity search against the disk-persisted swagger schema space."""
    print(f"\n[Search Query]: '{user_query}'")
    query_vector = get_embedding(user_query)
    #print(query_vector)

    print("Collection counts:", collection.count())
    if collection.count() == 0:
        print("No records in the collection. Run the indexing cell in Embeddings.ipynb first.")
        return

    results = collection.query(
        query_embeddings=[query_vector],
        n_results=1,
        include=["documents", "distances", "metadatas","embeddings"]
        
    )

    import pandas as pd
    pd.set_option('display.max_colwidth', None)  # Show full content in DataFrame
    df = pd.DataFrame(results["documents"][0], columns=["documents"])
    #print("\n[Search Results DataFrame]:")
    #print(df)
    df = pd.DataFrame(results["metadatas"][0], columns=["Metadata"])
    #print("\n[Search Results DataFrame]:")
    #print(df)
    df = pd.DataFrame(results["distances"][0], columns=["distances"])
    #print("\n[Search Results DataFrame]:")
    #print(df)
    



    # Display the results neatly
    if results and results["metadatas"] and results["metadatas"][0]:
        for i in range(len(results["metadatas"][0])):
            metadata = results["metadatas"][0][i]
            document = results["documents"][0][i]
            print(f"   Metadata Text: {metadata.get('endpoint', 'N/A')} ({metadata.get('summary', 'No summary')})")
            #print(f"-> Top Match Found: {metadata['endpoint']} ({metadata['summary']})")
            print(f"   Indexed Text: {document}")
    else:
        print("-> No matches found.")


# --- Engine Pipeline Execution ---
if __name__ == "__main__":
    results = collection.get()
    #for doc in results['documents']:
    #    print("\nDocument:", doc)
     
    search_openapi_catalog("show me deposits for the last 3 months")



[Search Query]: 'show me deposits for the last 3 months'
Collection counts: 7
   Metadata Text: POST /v1/queries/CheckDepositsATM (Receive User ATM Checks Deposits only)
   Indexed Text: Tags: Transactions, Check Deposits, ATM, User, Dashboard. Summary: Receive User ATM Checks Deposits only.Operational Action Details: Receive User ATM Checks Deposits only. Expected Client Input Parameters: field 'user_query' (string: Formatted and Validated User Query.), field 'CheckDepositsATM' (string: The textual Check Deposits ATM body content and other payload.). Returned Schema Parameters: field 'message_id' (string: Unique alphanumeric delivery tracking identifier.), field 'delivery_status' (string: Accepted or Rejected or Malformed).
